# META-CXR Full Evaluation — MedGemma + Spark (2x T4)

**Pipeline:**
- Phase 1 — Stage 1 BLIP inference (multiprocessing, 2 GPU)
- Phase 2 — MedGemma report generation (multiprocessing, 2 GPU)
- Phase 3 — NLP metrics: BLEU/METEOR/ROUGE-L via **Spark**, CIDEr/BERTScore/RadGraph/RadCliQ sequential
- Phase 4 — Clinical Efficacy: Precision / Recall / Macro F1
- Phase 5 — Assemble tables + upload to GCS

**Prerequisites:** Accelerator = GPU T4x2, Internet = on, Kaggle secrets: `GCS_SERVICE_ACCOUNT`, `HF_TOKEN`

## Cell 0 — Load Kaggle Secrets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

_s = UserSecretsClient()
os.environ['GCS_SERVICE_ACCOUNT'] = _s.get_secret('GCS_SERVICE_ACCOUNT')
os.environ['HF_TOKEN'] = _s.get_secret('HF_TOKEN')
os.environ['HUGGINGFACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
print('Secrets loaded: GCS_SERVICE_ACCOUNT, HF_TOKEN')

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'pyspark==3.5.1',
    'google-cloud-storage',
    'bert_score',
    'radgraph',
    'peft==0.10.0',
    'nltk',
    'pycocoevalcap',
    'transformers>=4.40.0',
    'accelerate',
    'omegaconf==2.3.0',
    'scikit-image',
    'torchinfo',
    'iopath',
    'hi-ml-multimodal',
    'timm>=0.9.0',
    'loralib==0.1.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages)

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
print('Dependencies ready.')

## Cell 2 — Clone Repo + env_config + Download Checkpoints

In [ ]:
import base64, json, os, subprocess
from pathlib import Path
from google.cloud import storage
from google.oauth2 import service_account

REPO_DIR       = Path('/kaggle/working/META-CXR')
GCS_PROJECT    = 'mimic-cxr-jpg-491409'
GCS_BUCKET     = 'meta-cxr-checkpoint'
CKPT_ROOT      = Path('/kaggle/temp/checkpoints')
STAGE1_DIR     = Path('/kaggle/temp/eval')
OUTPUT_DIR     = Path('/kaggle/working/eval_output')
FINAL_DIR      = OUTPUT_DIR / 'final'
VIS_ROOT       = '/kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite'
PROCESSED_ROOT = '/kaggle/input/datasets/phuong20052/mimic-cxr-p10-processed'

for d in [CKPT_ROOT, STAGE1_DIR, OUTPUT_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUNS = ['09_all_plus_raddino']


ENCODER_INFO = {
    '09_all_plus_raddino': {'RN50': '+', 'ViT': '-', 'Swin': '+', 'RadDINO': '+'},
}


def _build_gcs_client():
    raw = os.environ['GCS_SERVICE_ACCOUNT'].strip()
    try:
        info = json.loads(raw)
    except json.JSONDecodeError:
        info = json.loads(base64.b64decode(raw).decode())
    creds = service_account.Credentials.from_service_account_info(info)
    return storage.Client(project=GCS_PROJECT, credentials=creds)

gcs_client = _build_gcs_client()
bucket     = gcs_client.bucket(GCS_BUCKET)

# Clone repo
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone',
        'https://github.com/minhphuong150505/Meta-CXR-Kaggle.git', str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull'])
print(f'Repo ready: {REPO_DIR}')

# Generate env_config.yaml with Kaggle paths
java_home = subprocess.run(
    'readlink -f $(which java) | sed "s|/bin/java||"',
    shell=True, capture_output=True, text=True,
).stdout.strip() or '/usr/lib/jvm/java-8-openjdk-amd64'

env_cfg = (
    'paths:\n'
    f'  data_root: "{VIS_ROOT}"\n'
    f'  mimic_cxr_jpg_root: "{VIS_ROOT}"\n'
    f'  split_csv: "{VIS_ROOT}/mimic-cxr-2.0.0-split.csv"\n'
    '  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"\n'
    f'  chexpert_csv: "{VIS_ROOT}/mimic-cxr-2.0.0-chexpert.csv"\n'
    f'  metadata_csv: "{VIS_ROOT}/mimic-cxr-2.0.0-metadata.csv"\n'
    f'  processed_train_csv: "{PROCESSED_ROOT}/train.csv"\n'
    f'  processed_val_csv: "{PROCESSED_ROOT}/val.csv"\n'
    f'  processed_test_csv: "{PROCESSED_ROOT}/test.csv"\n'
    '  output_dir: "/kaggle/working/output"\n'
    'java:\n'
    f'  home: "{java_home}"\n'
    f'  path: "{java_home}/bin:"\n'
    'wandb:\n'
    '  entity: "phuong20052"\n'
    '  project: "meta-cxr-eval"\n'
)
(REPO_DIR / 'configs' / 'env_config.yaml').write_text(env_cfg)
print('env_config.yaml written.')

# Download checkpoints
for run in RUNS:
    dest = CKPT_ROOT / run / 'checkpoint_best.pth'
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'  cached: {run}'); continue
    blob = bucket.blob(f'{run}/checkpoint_best.pth')
    if blob.exists():
        blob.download_to_filename(str(dest))
        print(f'  downloaded: {run}')
    else:
        print(f'  NOT FOUND: {run}')
print('Setup complete.')

## Cell 3 — GPU Check + Split Runs Across GPUs

In [ ]:
import torch

n_gpus = torch.cuda.device_count()
print(f'GPUs: {n_gpus}')
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
assert n_gpus >= 1, 'Need at least 1 GPU'

# Round-robin split
GPU_SPLITS = {}
for i, run in enumerate(RUNS):
    gid = i % n_gpus
    GPU_SPLITS.setdefault(gid, []).append(run)

for gid, runs in GPU_SPLITS.items():
    print(f'  GPU {gid}: {runs}')

## Phase 1 — Stage 1 BLIP Inference (multiprocessing, 2 GPU)
Output: `/kaggle/temp/eval/stage1_cache/{run}_stage1.pt`

In [ ]:
# Write worker_stage1.py
worker1 = r'''
import argparse, gc, os, sys

p = argparse.ArgumentParser()
p.add_argument('--gpu',             type=int, required=True)
p.add_argument('--runs',            required=True)
p.add_argument('--checkpoint-root', required=True)
p.add_argument('--output-dir',      required=True)
p.add_argument('--sample-limit',    type=int, default=200)
p.add_argument('--num-workers',     type=int, default=2)
args = p.parse_args()

os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)  # must be before torch

REPO = '/kaggle/working/META-CXR'
sys.path.insert(0, REPO)
sys.path.insert(0, REPO + '/model')
os.chdir(REPO)

from pathlib import Path
from evaluation.eval_bertscore_medgemma_qformer import extract_stage1_run
import torch

ckpt_root  = Path(args.checkpoint_root)
output_dir = Path(args.output_dir)
runs       = [r.strip() for r in args.runs.split(',')]

for run in runs:
    print(f'[GPU {args.gpu}] Stage1 start: {run}', flush=True)
    try:
        records = extract_stage1_run(
            run, ckpt_root, output_dir,
            args.sample_limit, args.num_workers, reuse_cache=True,
        )
        print(f'[GPU {args.gpu}] Stage1 OK: {run} ({len(records)} records)', flush=True)
    except Exception as e:
        print(f'[GPU {args.gpu}] Stage1 FAIL: {run} {e}', flush=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'[GPU {args.gpu}] Phase1 done.', flush=True)
'''
with open('/kaggle/working/worker_stage1.py', 'w') as f:
    f.write(worker1)
print('worker_stage1.py written.')

In [ ]:
import subprocess, sys
from pathlib import Path

procs1, logs1 = {}, {}
for gid, runs in GPU_SPLITS.items():
    log = Path(f'/kaggle/working/stage1_gpu{gid}.log')
    lf  = open(log, 'w')
    cmd = [
        sys.executable, '/kaggle/working/worker_stage1.py',
        '--gpu',             str(gid),
        '--runs',            ','.join(runs),
        '--checkpoint-root', str(CKPT_ROOT),
        '--output-dir',      str(STAGE1_DIR),
        '--sample-limit',    '200',
        '--num-workers',     '2',
    ]
    procs1[gid] = subprocess.Popen(cmd, stdout=lf, stderr=lf)
    logs1[gid]  = log
    print(f'Launched GPU {gid}: {runs}')

print('Waiting for Phase 1...')
for gid, proc in procs1.items():
    proc.wait()
    print(f'  GPU {gid} rc={proc.returncode}')
    if proc.returncode != 0:
        print(open(logs1[gid]).read()[-2000:])

for run in RUNS:
    ok = (STAGE1_DIR / 'stage1_cache' / f'{run}_stage1.pt').exists()
    print(f'  {run}: {"OK" if ok else "MISSING"}')

## Phase 2 — MedGemma Report Generation (multiprocessing, 2 GPU)

Mỗi GPU load 1 MedGemma instance (~9 GB). BERTScore tính ngay trong worker.  
Output: `eval_output/reports_medgemma_qformer_{run}.jsonl`

In [ ]:
worker2 = r'''
import argparse, gc, json, os, sys

p = argparse.ArgumentParser()
p.add_argument('--gpu',            type=int, required=True)
p.add_argument('--runs',           required=True)
p.add_argument('--stage1-dir',     required=True)
p.add_argument('--output-dir',     required=True)
p.add_argument('--max-new-tokens', type=int, default=300)
args = p.parse_args()

os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)  # must be before torch

REPO = '/kaggle/working/META-CXR'
sys.path.insert(0, REPO)
sys.path.insert(0, REPO + '/model')
os.chdir(REPO)

import torch
from pathlib import Path
from evaluation.eval_bertscore_medgemma_qformer import (
    MedGemmaQFormerGenerator,
    run_generation_for_records,
    MEDGEMMA_MODEL_ID,
    MEDGEMMA_LORA_ID,
)

stage1_dir = Path(args.stage1_dir)
output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
runs = [r.strip() for r in args.runs.split(',')]

print(f'[GPU {args.gpu}] Loading MedGemma...', flush=True)
generator = MedGemmaQFormerGenerator(
    MEDGEMMA_MODEL_ID, MEDGEMMA_LORA_ID, device_map_auto=False
)

results = {}
for run in runs:
    cache = stage1_dir / 'stage1_cache' / f'{run}_stage1.pt'
    if not cache.exists():
        print(f'[GPU {args.gpu}] SKIP {run}: no cache', flush=True)
        continue
    records = torch.load(str(cache), map_location='cpu')
    print(f'[GPU {args.gpu}] Stage2: {run} ({len(records)} records)', flush=True)
    bs, n = run_generation_for_records(
        generator, records, run, output_dir, args.max_new_tokens
    )
    results[run] = {'bertscore': round(bs, 4), 'n_samples': n}
    print(f'[GPU {args.gpu}] {run}: BERTScore={bs:.4f}', flush=True)

with open(output_dir / f'bertscore_gpu{args.gpu}.json', 'w') as f:
    json.dump(results, f, indent=2)

del generator
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'[GPU {args.gpu}] Phase2 done.', flush=True)
'''
with open('/kaggle/working/worker_stage2.py', 'w') as f:
    f.write(worker2)
print('worker_stage2.py written.')

In [ ]:
procs2, logs2 = {}, {}
for gid, runs in GPU_SPLITS.items():
    log = Path(f'/kaggle/working/stage2_gpu{gid}.log')
    lf  = open(log, 'w')
    cmd = [
        sys.executable, '/kaggle/working/worker_stage2.py',
        '--gpu',            str(gid),
        '--runs',           ','.join(runs),
        '--stage1-dir',     str(STAGE1_DIR),
        '--output-dir',     str(OUTPUT_DIR),
        '--max-new-tokens', '300',
    ]
    procs2[gid] = subprocess.Popen(cmd, stdout=lf, stderr=lf)
    logs2[gid]  = log
    print(f'Launched GPU {gid}: {runs}')

print('Waiting for Phase 2 (~30-60 min)...')
for gid, proc in procs2.items():
    proc.wait()
    print(f'  GPU {gid} rc={proc.returncode}')
    if proc.returncode != 0:
        print(open(logs2[gid]).read()[-2000:])

# Collect BERTScore from worker result files
bertscore_by_run = {}
available_runs   = []
for gid in GPU_SPLITS:
    result_path = OUTPUT_DIR / f'bertscore_gpu{gid}.json'
    if result_path.exists():
        with open(result_path) as f:
            bertscore_by_run.update(json.load(f))
for run in RUNS:
    if (OUTPUT_DIR / f'reports_medgemma_qformer_{run}.jsonl').exists():
        available_runs.append(run)

print(f'\nJSONL ready: {available_runs}')
for run, v in bertscore_by_run.items():
    print(f'  {run}: BERTScore={v["bertscore"]}')

## Phase 3a — BLEU/METEOR/ROUGE-L via Spark (per-sample parallel)

In [ ]:
import os, re
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql.functions import col, avg, count, pandas_udf

N_CORES = os.cpu_count() or 2
spark = (
    SparkSession.builder
    .master(f'local[{N_CORES}]')
    .appName('MetaCXR-NLGEval')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', str(N_CORES * 4))
    .getOrCreate()
)
print(f'Spark {spark.version} — local[{N_CORES}]')

rows = []
for run in available_runs:
    with open(OUTPUT_DIR / f'reports_medgemma_qformer_{run}.jsonl') as f:
        for sid, line in enumerate(f):
            obj = json.loads(line)
            rows.append((run, sid, str(obj.get('pred', '')), str(obj.get('ref', ''))))

schema = StructType([
    StructField('run',       StringType(),  False),
    StructField('sample_id', IntegerType(), False),
    StructField('pred',      StringType(),  True),
    StructField('ref',       StringType(),  True),
])
df = spark.createDataFrame(rows, schema=schema).repartition(N_CORES * 4)
print(f'Rows: {df.count()}')

def _tok(t): return re.findall(r'\w+', str(t).lower())

@pandas_udf(FloatType())
def bleu1_udf(ps: pd.Series, rs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    sm = SmoothingFunction().method1
    return pd.Series([float(sentence_bleu([_tok(r)], _tok(p), weights=(1,0,0,0), smoothing_function=sm)) if _tok(p) else 0.0 for p, r in zip(ps, rs)])

@pandas_udf(FloatType())
def bleu2_udf(ps: pd.Series, rs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    sm = SmoothingFunction().method1
    return pd.Series([float(sentence_bleu([_tok(r)], _tok(p), weights=(.5,.5,0,0), smoothing_function=sm)) if _tok(p) else 0.0 for p, r in zip(ps, rs)])

@pandas_udf(FloatType())
def bleu3_udf(ps: pd.Series, rs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    sm = SmoothingFunction().method1
    return pd.Series([float(sentence_bleu([_tok(r)], _tok(p), weights=(1/3,1/3,1/3,0), smoothing_function=sm)) if _tok(p) else 0.0 for p, r in zip(ps, rs)])

@pandas_udf(FloatType())
def bleu4_udf(ps: pd.Series, rs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    sm = SmoothingFunction().method1
    return pd.Series([float(sentence_bleu([_tok(r)], _tok(p), weights=(.25,.25,.25,.25), smoothing_function=sm)) if _tok(p) else 0.0 for p, r in zip(ps, rs)])

@pandas_udf(FloatType())
def meteor_udf(ps: pd.Series, rs: pd.Series) -> pd.Series:
    from nltk.translate.meteor_score import meteor_score
    out = []
    for p, r in zip(ps, rs):
        pt, rt = _tok(p), _tok(r)
        try: out.append(float(meteor_score([rt], pt)) if pt and rt else 0.0)
        except: out.append(0.0)
    return pd.Series(out)

@pandas_udf(FloatType())
def rougel_udf(ps: pd.Series, rs: pd.Series) -> pd.Series:
    def _lcs(a, b):
        prev = [0]*(len(b)+1)
        for ai in a:
            curr = [0]*(len(b)+1)
            for j, bj in enumerate(b, 1):
                curr[j] = prev[j-1]+1 if ai==bj else max(prev[j], curr[j-1])
            prev = curr
        return prev[len(b)]
    out = []
    for p, r in zip(ps, rs):
        pt, rt = _tok(p), _tok(r)
        if not pt or not rt: out.append(0.0); continue
        lcs = _lcs(pt, rt)
        pr, rec = lcs/len(pt), lcs/len(rt)
        out.append(float(2*pr*rec/(pr+rec)) if pr+rec > 0 else 0.0)
    return pd.Series(out)

nlg_spark = (
    df
    .withColumn('bleu1',  bleu1_udf(col('pred'), col('ref')))
    .withColumn('bleu2',  bleu2_udf(col('pred'), col('ref')))
    .withColumn('bleu3',  bleu3_udf(col('pred'), col('ref')))
    .withColumn('bleu4',  bleu4_udf(col('pred'), col('ref')))
    .withColumn('meteor', meteor_udf(col('pred'), col('ref')))
    .withColumn('rougel', rougel_udf(col('pred'), col('ref')))
    .groupBy('run')
    .agg(
        avg('bleu1').alias('BLEU-1'), avg('bleu2').alias('BLEU-2'),
        avg('bleu3').alias('BLEU-3'), avg('bleu4').alias('BLEU-4'),
        avg('meteor').alias('METEOR'), avg('rougel').alias('ROUGE-L'),
        count('*').alias('n_samples'),
    )
    .orderBy('run')
    .toPandas()
)
for c in ['BLEU-1','BLEU-2','BLEU-3','BLEU-4','METEOR','ROUGE-L']:
    nlg_spark[c] = nlg_spark[c].round(4)

spark.stop()
print(nlg_spark.to_string(index=False))

## Phase 3b — CIDEr + RadGraph F1 + RadCliQ (sequential)

In [ ]:
import numpy as np
from pycocoevalcap.cider.cider import Cider
from radgraph import F1RadGraph

print('Loading radgraph-xl (CPU)...')
f1radgraph = F1RadGraph(reward_level='all', model_type='radgraph-xl')

cider_by_run    = {}
radgraph_by_run = {}
radcliq_by_run  = {}

for run in available_runs:
    preds, refs, pm, rm = [], [], {}, {}
    with open(OUTPUT_DIR / f'reports_medgemma_qformer_{run}.jsonl') as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            p, r = str(obj.get('pred', '')), str(obj.get('ref', ''))
            preds.append(p); refs.append(r)
            pm[i] = [p]; rm[i] = [r]

    cider_score, _ = Cider().compute_score(rm, pm)
    cider_by_run[run] = round(float(cider_score), 4)

    vp = [p for p, r in zip(preds, refs) if p.strip() and r.strip()]
    vr = [r for p, r in zip(preds, refs) if p.strip() and r.strip()]
    if vp:
        rwd, _, _, _ = f1radgraph(hyps=vp, refs=vr)
        radgraph_by_run[run] = {
            'entity_f1':   round(float(rwd[0]), 4),
            'relation_f1': round(float(rwd[1]), 4),
            'overall_f1':  round(float(rwd[2]), 4),
        }
    else:
        radgraph_by_run[run] = {'entity_f1': 0.0, 'relation_f1': 0.0, 'overall_f1': 0.0}

    bs = bertscore_by_run.get(run, {}).get('bertscore', float('nan'))
    rg = radgraph_by_run[run]['overall_f1']
    radcliq_by_run[run] = round(float(np.sqrt((1-bs)**2 + (1-rg)**2)), 4) if not np.isnan(bs) else 'N/A'

    print(f'  {run}: CIDEr={cider_by_run[run]}, RadGraph={rg}, RadCliQ={radcliq_by_run[run]}')

## Phase 4 — Clinical Efficacy (Precision / Recall / Macro F1)

BLIP classification head của `05_biovil_swin_raddino`, đánh giá trên 5 abnormalities phổ biến.

In [ ]:
import sys, os, torch, numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score
from types import SimpleNamespace

REPO_S = str(REPO_DIR)
for p in [REPO_S, REPO_S + '/model']:
    if p not in sys.path: sys.path.insert(0, p)
os.chdir(REPO_S)

from model.lavis.common.config import Config
from model.lavis.tasks import setup_task
from local_config import VIS_ROOT as _VIS
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset

ABNORMALITIES_14 = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
]
COMMON_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

run_ce   = '05_biovil_swin_raddino'
cfg      = Config(SimpleNamespace(
    cfg_path=str(REPO_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_ce}.yaml'),
    options=None,
))
model_ce = setup_task(cfg).build_model(cfg)
ckpt     = torch.load(str(CKPT_ROOT / run_ce / 'checkpoint_best.pth'), map_location='cpu', weights_only=False)
model_ce.load_state_dict(ckpt.get('model', ckpt), strict=False)
model_ce.to(DEVICE).eval()

dataset = MIMIC_CXR_Dataset(
    vis_processor=None, text_processor=None,
    vis_root=_VIS, split='test', cfg=cfg, truncate=None,
)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

all_logits, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(loader, desc='Clinical Efficacy'):
        logits, _ = model_ce.forward_image(batch['image'].to(DEVICE, non_blocking=True))
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(batch['classification_labels'].float().cpu().numpy())

logits_arr = np.concatenate(all_logits)
labels_arr = np.concatenate(all_labels)
preds_arr  = np.argmax(torch.softmax(torch.tensor(logits_arr), dim=-1).numpy(), axis=-1)

per_abn = {}
f1s, precs, recs = [], [], []
for i, abn in enumerate(ABNORMALITIES_14):
    if abn == 'No Finding': continue
    yt = (labels_arr[:, i] == 1).astype(int)
    yp = (preds_arr[:, i] == 1).astype(int)
    pr = precision_score(yt, yp, zero_division=0)
    rc = recall_score(yt, yp, zero_division=0)
    f  = f1_score(yt, yp, zero_division=0)
    per_abn[abn] = {'precision': round(pr,4), 'recall': round(rc,4), 'f1': round(f,4)}
    if abn in COMMON_5:
        f1s.append(f); precs.append(pr); recs.append(rc)

clinical_efficacy = {
    'precision': round(float(np.mean(precs)), 4),
    'recall':    round(float(np.mean(recs)),  4),
    'macro_f1':  round(float(np.mean(f1s)),   4),
    'per_abnormality': per_abn,
}
del model_ce; torch.cuda.empty_cache()

print(f'Precision = {clinical_efficacy["precision"]}')
print(f'Recall    = {clinical_efficacy["recall"]}')
print(f'Macro F1  = {clinical_efficacy["macro_f1"]}')

## Phase 5 — Assemble Tables + Upload GCS

In [ ]:
import pandas as pd, json

rows_nlg = []
nlg_idx  = nlg_spark.set_index('run')
for run in available_runs:
    row = {'Run': run, **ENCODER_INFO[run]}
    if run in nlg_idx.index:
        r = nlg_idx.loc[run]
        for k in ['BLEU-1','BLEU-2','BLEU-3','BLEU-4','METEOR','ROUGE-L','n_samples']:
            row[k] = r[k]
    row['CIDEr']       = cider_by_run.get(run, 'N/A')
    row['BERTScore']   = bertscore_by_run.get(run, {}).get('bertscore', 'N/A')
    row['RadGraph_F1'] = radgraph_by_run.get(run, {}).get('overall_f1', 'N/A')
    row['RadCliQ']     = radcliq_by_run.get(run, 'N/A')
    rows_nlg.append(row)

table_nlg = pd.DataFrame(rows_nlg)

print('=== NLG + NLP-Derived Clinical Metrics ===')
cols = ['Run','RN50','ViT','Swin','BLEU-1','BLEU-4','METEOR','ROUGE-L','CIDEr','BERTScore','RadGraph_F1','RadCliQ']
print(table_nlg[cols].to_string(index=False))
print('\n=== Clinical Efficacy ===')
print(f'  Precision={clinical_efficacy["precision"]}  Recall={clinical_efficacy["recall"]}  Macro F1={clinical_efficacy["macro_f1"]}')

# Save
nlg_csv     = FINAL_DIR / 'nlg_metrics_medgemma.csv'
summary_j   = FINAL_DIR / 'eval_summary_medgemma.json'
efficacy_j  = FINAL_DIR / 'clinical_efficacy_05_biovil_swin_raddino.json'

table_nlg.to_csv(nlg_csv, index=False)
with open(summary_j, 'w') as f:
    json.dump({
        'llm': 'google/medgemma-1.5-4b-it',
        'compute': f'Kaggle 2xT4, Spark local[{N_CORES}]',
        'runs': rows_nlg,
        'clinical_efficacy': clinical_efficacy,
    }, f, indent=2)
with open(efficacy_j, 'w') as f:
    json.dump({'run': '05_biovil_swin_raddino', **clinical_efficacy}, f, indent=2)

# Upload to GCS
GCS_OUT = 'eval/spark_medgemma_eval'

def _up(local_path):
    bname = f'{GCS_OUT}/{Path(local_path).name}'
    bucket.blob(bname).upload_from_filename(str(local_path))
    print(f'  gs://{GCS_BUCKET}/{bname}')

print('\nUploading to GCS...')
for p in [nlg_csv, summary_j, efficacy_j]: _up(p)

for run in available_runs:
    jl = OUTPUT_DIR / f'reports_medgemma_qformer_{run}.jsonl'
    if jl.exists():
        bname = f'{GCS_OUT}/jsonl/{jl.name}'
        bucket.blob(bname).upload_from_filename(str(jl))
        print(f'  gs://{GCS_BUCKET}/{bname}')

print('Done.')